# Text Generation using Vanilla RNN, LSTM, and GRU

**Problem Statement:** Design and implement a DL model capable of learning the underlying structure, grammar, and contextual dependencies of a given text corpus to generate coherent and meaningful text sequences using Vanilla RNN, LSTM, and GRU.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

## Loading the text corpus
Using lyrics from Billie Jean by Michael Jackson.

In [ ]:
corpus = '''
she was more like a beauty queen from a movie scene
i said dont mind but what do you mean i am the one
who will dance on the floor in the round
she said i am the one who will dance on the floor in the round
she told me her name was billie jean as she caused a scene
then every head turned with eyes that dreamed of being the one
who will dance on the floor in the round
people always told me be careful of what you do
and dont go around breaking young girls hearts
and mother always told me be careful of who you love
and be careful of what you do cause the lie becomes the truth
billie jean is not my lover
she is just a girl who claims that i am the one
but the kid is not my son
she says i am the one but the kid is not my son
for forty days and forty nights the law was on her side
but who can stand when she is in demand
her schemes and plans cause we danced on the floor in the round
so take my strong advice just remember to always think twice
she told my baby we danced till three then she looked at me
then showed a photo my baby cried his eyes were like mine
billie jean is not my lover
she is just a girl who claims that i am the one
but the kid is not my son
she says i am the one but the kid is not my son
'''

print(corpus)

## Tokenization and Sequence Creation

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

# creating n-gram sequences
input_sequences = []
for line in corpus.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

## Model 1 — Vanilla RNN

In [ ]:
rnn_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    SimpleRNN(64),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_history = rnn_model.fit(X, y, epochs=100, verbose=0)
print("RNN done — Loss:", round(rnn_history.history['loss'][-1], 4), "Acc:", round(rnn_history.history['accuracy'][-1], 4))

## Model 2 — LSTM

In [ ]:
lstm_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_history = lstm_model.fit(X, y, epochs=100, verbose=0)
print("LSTM done — Loss:", round(lstm_history.history['loss'][-1], 4), "Acc:", round(lstm_history.history['accuracy'][-1], 4))

## Model 3 — GRU

In [ ]:
gru_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    GRU(64),
    Dense(total_words, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_history = gru_model.fit(X, y, epochs=100, verbose=0)
print("GRU done — Loss:", round(gru_history.history['loss'][-1], 4), "Acc:", round(gru_history.history['accuracy'][-1], 4))

## Comparing Training Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(rnn_history.history['loss'], label='RNN')
plt.plot(lstm_history.history['loss'], label='LSTM')
plt.plot(gru_history.history['loss'], label='GRU')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.show()

## Text Generation Function

In [ ]:
def generate_text(model, seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

## Generating Text — 5 words

In [ ]:
seed = "she told me"

print("RNN :", generate_text(rnn_model, seed, 5))
print("LSTM:", generate_text(lstm_model, seed, 5))
print("GRU :", generate_text(gru_model, seed, 5))

---
# Student Learning Tasks

## Task 1 — Replace corpus with your own paragraph
Already done — replaced the default corpus with Billie Jean lyrics by Michael Jackson.

## Task 2 — Increase embedding dimension (32 → 64)
Testing with LSTM to see if bigger embeddings help.

In [ ]:
lstm_emb64 = Sequential([
    Embedding(total_words, 64, input_length=max_len-1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])
lstm_emb64.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_emb64_history = lstm_emb64.fit(X, y, epochs=100, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(lstm_history.history['loss'], label='emb=32')
plt.plot(lstm_emb64_history.history['loss'], label='emb=64')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LSTM — Embedding 32 vs 64')
plt.legend()
plt.show()

## Task 3 — Increase epochs (100 → 200)

In [ ]:
lstm_200 = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])
lstm_200.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_200_history = lstm_200.fit(X, y, epochs=200, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(lstm_history.history['loss'], label='100 epochs')
plt.plot(lstm_200_history.history['loss'], label='200 epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LSTM — 100 vs 200 Epochs')
plt.legend()
plt.show()

## Task 4 — Change hidden units (64 → 128)

In [ ]:
lstm_128 = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    LSTM(128),
    Dense(total_words, activation='softmax')
])
lstm_128.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_128_history = lstm_128.fit(X, y, epochs=100, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(lstm_history.history['loss'], label='64 units')
plt.plot(lstm_128_history.history['loss'], label='128 units')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LSTM — 64 vs 128 Hidden Units')
plt.legend()
plt.show()

## Task 5 — Generate 10 words instead of 5

In [ ]:
seed = "she told me"

print("RNN  (10 words):", generate_text(rnn_model, seed, 10))
print("LSTM (10 words):", generate_text(lstm_model, seed, 10))
print("GRU  (10 words):", generate_text(gru_model, seed, 10))

## Final Comparison — Accuracy

In [ ]:
models = ['RNN', 'LSTM', 'GRU']
accs = [rnn_history.history['accuracy'][-1], lstm_history.history['accuracy'][-1], gru_history.history['accuracy'][-1]]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accs, color=['#e74c3c', '#3498db', '#2ecc71'])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{acc:.2%}', ha='center')
plt.ylabel('Accuracy')
plt.title('Final Training Accuracy (emb=32, units=64, 100 epochs)')
plt.ylim(0, 1.1)
plt.show()

## Conclusion

- **Vanilla RNN** learns short patterns but struggles with longer sentences because of vanishing gradients.
- **LSTM** captures long-range dependencies better because of its forget, input, and output gates.
- **GRU** gives similar performance to LSTM but trains faster since it has only 2 gates instead of 3.
- Increasing **embedding dimension** (32 → 64) gave slightly better loss since the model gets richer word representations.
- Increasing **epochs** (100 → 200) let the model converge more but risks overfitting on a small corpus.
- Increasing **hidden units** (64 → 128) gave more capacity to learn complex patterns.
- Generating **10 words** showed that LSTM and GRU maintain more coherent output over longer sequences compared to vanilla RNN.